# UIT DSC 2026 LegalIR - Step 7b.1 Clean SBERT-CL Train

Train lại `bkai-foundation-models/vietnamese-bi-encoder` trên đúng Step 4 split/chunks, dùng PyVi word segmentation và negatives từ Step 5 fused candidates. Notebook này chỉ tạo checkpoint sạch cho Step 7b.2 global dense retrieval; chưa tạo submission.

Contract: fail-fast nếu thiếu artifact bắt buộc, không tự chuyển sang candidate pool hoặc checkpoint khác.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'sentence-transformers>=3.0,<6', 'pyvi', 'safetensors', 'tqdm'], check=True)


## Config And Required Kaggle Inputs

Expected dataset layout:

```text
/kaggle/input/datasets/bowboochua9/stnhdscduaiti26/
|-- step4/
|   |-- chunks.jsonl
|   |-- train_split.json
|   `-- dev_split.json
|-- step6/
    `-- step5/
        `-- rankings/
            |-- train_rankings_step5_fused.jsonl
            `-- dev_rankings_step5_fused.jsonl
```


In [ ]:
from __future__ import annotations

import gc, hashlib, json, math, os, random, re, shutil, time, unicodedata, zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import torch
import torch.nn.functional as F
from pyvi.ViTokenizer import tokenize as pyvi_tokenize
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
STEP4_DIR = DATA_ROOT / 'step4'
# Step5 fused rankings are stored inside the Step6 artifact bundle in the
# current Kaggle dataset. This is the exact Step5 artifact path, not a fallback.
STEP5_ARTIFACT_DIR = DATA_ROOT / 'step6' / 'step5'
CHUNKS_FILE = STEP4_DIR / 'chunks.jsonl'
TRAIN_FILE = STEP4_DIR / 'train_split.json'
DEV_FILE = STEP4_DIR / 'dev_split.json'
TRAIN_RANKINGS_FILE = STEP5_ARTIFACT_DIR / 'rankings' / 'train_rankings_step5_fused.jsonl'
DEV_RANKINGS_FILE = STEP5_ARTIFACT_DIR / 'rankings' / 'dev_rankings_step5_fused.jsonl'

OUTPUT_ROOT = Path('/kaggle/working/step7b')
OUTPUT_DIR = OUTPUT_ROOT / 'checkpoint1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'bkai-foundation-models/vietnamese-bi-encoder'
ALLOWED_MODELS = {
    'BAAI/bge-m3',
    'bkai-foundation-models/vietnamese-bi-encoder',
    'itdainb/PhoRanker',
    'Qwen/Qwen3-Reranker-0.6B',
    'BAAI/bge-reranker-v2-m3',
    'AITeamVN/Vietnamese_Reranker',
    'bqbbao6/vietnamese-legal-embedding',
    'AITeamVN/Vietnamese_Embedding_v2',
    'Qwen/Qwen3-Embedding-0.6B',
}

def preview_tree(root: Path, max_depth: int = 3, max_items: int = 120) -> list[str]:
    lines = []
    if not root.exists():
        return [f'{root} [missing]']
    root = root.resolve()
    for i, path in enumerate(sorted(root.rglob('*'))):
        if i >= max_items:
            lines.append(f'... truncated after {max_items} items')
            break
        rel = path.relative_to(root)
        if len(rel.parts) > max_depth:
            continue
        suffix = '/' if path.is_dir() else ''
        lines.append(str(rel).replace('\\', '/') + suffix)
    return lines

required = {
    'chunks': CHUNKS_FILE,
    'train_split': TRAIN_FILE,
    'dev_split': DEV_FILE,
    'step5_train_rankings': TRAIN_RANKINGS_FILE,
    'step5_dev_rankings': DEV_RANKINGS_FILE,
}
missing_required = {name: str(path) for name, path in required.items() if not path.exists()}
if missing_required:
    print('Missing required inputs:', json.dumps(missing_required, ensure_ascii=False, indent=2))
    print('\n/kaggle/input preview:')
    print('\n'.join(preview_tree(Path('/kaggle/input'), max_depth=2, max_items=80)))
    print('\nDATA_ROOT preview:')
    print('\n'.join(preview_tree(DATA_ROOT, max_depth=4, max_items=160)))
    raise AssertionError('Missing required Step 7b.1 inputs. Dataset must contain step4/chunks.jsonl, step4/train_split.json, step4/dev_split.json, and step6/step5/rankings/*.jsonl under DATA_ROOT exactly.')
assert MODEL_NAME in ALLOWED_MODELS, f'Model not whitelisted: {MODEL_NAME}'

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## Utilities, Loaders, And Metrics


In [ ]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def iter_jsonl(path: Path) -> Iterable[dict[str, Any]]:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def strip_accents(text: str) -> str:
    text = unicodedata.normalize('NFD', str(text))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', text)

TOKEN_RE = re.compile(r'\w+', flags=re.UNICODE)

def lexical_tokens(text: str) -> list[str]:
    return TOKEN_RE.findall(strip_accents(str(text).lower()))

def pyvi_segment(text: str) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return pyvi_tokenize(text) if text else ''

def normalize_text_for_encoder(text: str, max_chars: int = 5000) -> str:
    return re.sub(r'\s+', ' ', str(text)).strip()[:max_chars]

def gold_docs(row: dict[str, Any]) -> list[str]:
    return [str(x) for x in row.get('answer', [])]

def evaluate_rankings(rankings: dict[str, list[str]], payload: dict[str, Any], ks=(1, 5, 20, 50, 90, 100)) -> dict[str, Any]:
    macro = defaultdict(float)
    n = 0
    for qid, row in payload.items():
        qid = str(qid)
        gold = gold_docs(row)
        gold_set = set(gold)
        pred = [str(x) for x in rankings.get(qid, [])]
        n += 1
        first_hit = None
        for idx, doc_id in enumerate(pred, start=1):
            if doc_id in gold_set:
                first_hit = idx
                break
        for k in ks:
            top = pred[:k]
            hits = len(set(top) & gold_set)
            macro[f'recall@{k}'] += hits / max(1, len(gold_set))
            macro[f'precision@{k}'] += hits / max(1, k)
            macro[f'hit@{k}'] += 1.0 if hits else 0.0
        macro['mrr'] += 0.0 if first_hit is None else 1.0 / first_hit
    return {'macro': {k: v / max(1, n) for k, v in sorted(macro.items())}, 'num_queries': n}

def load_chunks(path: Path) -> tuple[list[dict[str, Any]], dict[str, list[int]], set[str]]:
    chunks = []
    doc_to_chunk_indices = defaultdict(list)
    valid_doc_ids = set()
    for idx, row in enumerate(iter_jsonl(path)):
        doc_id = str(row.get('doc_id', ''))
        chunk = {
            'chunk_idx': idx,
            'chunk_id': str(row.get('chunk_id', idx)),
            'doc_id': doc_id,
            'text': str(row.get('text', '')),
            'heading': str(row.get('heading', '')),
            'word_count': int(row.get('word_count') or 0),
        }
        chunks.append(chunk)
        if doc_id:
            doc_to_chunk_indices[doc_id].append(idx)
            valid_doc_ids.add(doc_id)
        if (idx + 1) % 50000 == 0:
            print(f'loaded {idx + 1:,} chunks')
    return chunks, doc_to_chunk_indices, valid_doc_ids

def load_fused_rankings(path: Path) -> dict[str, list[str]]:
    rankings = {}
    for row in iter_jsonl(path):
        qid = str(row.get('query_id', ''))
        if not qid or 'fused_doc_ids' not in row:
            raise ValueError(f'Bad ranking row in {path}: {row.keys()}')
        rankings[qid] = [str(x) for x in row['fused_doc_ids']]
    return rankings


## Load Data And Build Supervision Skeleton


In [ ]:
@dataclass(frozen=True)
class Step7b1Config:
    model_name: str = MODEL_NAME
    seed: int = SEED
    max_seq_length: int = 256
    frozen_encode_batch_size: int = 128
    train_batch_size: int = 8
    eval_batch_size: int = 128
    epochs: int = 3
    learning_rate: float = 2e-5
    weight_decay: float = 0.02
    warmup_ratio: float = 0.1
    temperature: float = 0.05
    positives_per_gold: int = 1
    positive_lexical_top_n: int = 5
    candidate_docs_per_query: int = 80
    lexical_negative_band_start: int = 3
    lexical_negative_band_end: int = 40
    semantic_negative_pool_size: int = 40
    eval_candidate_docs: int = 100
    eval_evidence_chunks_per_doc: int = 1
    max_text_chars: int = 5000
    best_metric: str = 'recall@5'

config = Step7b1Config()
write_json(OUTPUT_DIR / 'configs' / 'step7b1_config.json', asdict(config))

train_payload = read_json(TRAIN_FILE)
dev_payload = read_json(DEV_FILE)
chunks, doc_to_chunk_indices, valid_doc_ids = load_chunks(CHUNKS_FILE)
train_rankings = load_fused_rankings(TRAIN_RANKINGS_FILE)
dev_rankings = load_fused_rankings(DEV_RANKINGS_FILE)

missing_train = sorted(set(map(str, train_payload)) - set(train_rankings))
missing_dev = sorted(set(map(str, dev_payload)) - set(dev_rankings))
assert not missing_train, f'Missing train rankings for {len(missing_train)} queries, first={missing_train[:5]}'
assert not missing_dev, f'Missing dev rankings for {len(missing_dev)} queries, first={missing_dev[:5]}'
print(f'chunks={len(chunks):,}; docs={len(valid_doc_ids):,}; train={len(train_payload):,}; dev={len(dev_payload):,}')

def lexical_overlap_score(question: str, text: str, heading: str = '') -> float:
    q_counter = Counter(lexical_tokens(question))
    if not q_counter:
        return 0.0
    q_set = set(q_counter)
    c_counter = Counter(lexical_tokens(text[:6000]))
    overlap = sum(min(q_counter[tok], c_counter[tok]) for tok in q_set)
    heading_norm = strip_accents(heading.lower())
    heading_bonus = 0.20 if any(tok in heading_norm for tok in q_set) else 0.0
    return overlap / max(1, len(q_set)) + heading_bonus

def top_lexical_chunks_for_doc(question: str, doc_id: str, top_n: int) -> list[int]:
    scored = []
    for chunk_idx in doc_to_chunk_indices.get(str(doc_id), []):
        chunk = chunks[chunk_idx]
        if not chunk['text'].strip():
            continue
        score = lexical_overlap_score(question, chunk['text'], chunk['heading'])
        scored.append((score, -abs(chunk['word_count'] - 320), -chunk_idx, chunk_idx))
    scored.sort(reverse=True)
    return [chunk_idx for *_unused, chunk_idx in scored[:top_n]]

def build_structures(payload: dict[str, Any], rankings: dict[str, list[str]], include_negatives: bool):
    by_query = {}
    needed = set()
    missing_positive_docs = 0
    empty_negative_queries = 0
    for qid, row in tqdm(list(payload.items()), desc='Build supervision skeleton'):
        qid = str(qid)
        question = str(row.get('question', ''))
        gold = gold_docs(row)
        gold_set = set(gold)
        positives_by_doc = {}
        for doc_id in gold:
            pos = top_lexical_chunks_for_doc(question, doc_id, config.positive_lexical_top_n)
            if not pos:
                missing_positive_docs += 1
                continue
            positives_by_doc[doc_id] = pos
            needed.update(pos)
        negs = []
        if include_negatives:
            for cand_doc in rankings[qid][:config.candidate_docs_per_query]:
                cand_doc = str(cand_doc)
                if cand_doc in gold_set:
                    continue
                cand_chunks = top_lexical_chunks_for_doc(question, cand_doc, 1)
                if cand_chunks:
                    negs.append(cand_chunks[0])
                    needed.add(cand_chunks[0])
            if not negs:
                empty_negative_queries += 1
        by_query[qid] = {'question': question, 'gold': gold, 'positives_by_doc': positives_by_doc, 'negative_candidates': negs}
    return by_query, needed, {'num_queries': len(payload), 'missing_positive_docs': missing_positive_docs, 'empty_negative_queries': empty_negative_queries, 'needed_unique_chunks': len(needed), 'include_negatives': include_negatives}

train_struct, train_needed, train_skeleton_report = build_structures(train_payload, train_rankings, True)
dev_struct, dev_needed, dev_skeleton_report = build_structures(dev_payload, dev_rankings, False)
needed_chunk_indices = sorted(train_needed | dev_needed)
write_json(OUTPUT_DIR / 'training' / 'supervision_skeleton_report.json', {'train': train_skeleton_report, 'dev': dev_skeleton_report, 'total_needed_unique_chunks': len(needed_chunk_indices)})
print(json.dumps({'train': train_skeleton_report, 'dev': dev_skeleton_report, 'total_needed_unique_chunks': len(needed_chunk_indices)}, ensure_ascii=False, indent=2))


## PyVi Segmentation And Frozen BKAI Mining


In [ ]:
SEGMENTED_CHUNKS_FILE = OUTPUT_DIR / 'cache' / 'segmented_needed_chunks.jsonl'
SEGMENTED_QUERIES_FILE = OUTPUT_DIR / 'cache' / 'segmented_queries.json'

if SEGMENTED_CHUNKS_FILE.exists():
    segment_lookup = {int(row['chunk_idx']): row['segmented_text'] for row in iter_jsonl(SEGMENTED_CHUNKS_FILE)}
    missing = [idx for idx in needed_chunk_indices if idx not in segment_lookup]
    assert not missing, f'Segmented chunk cache incomplete: {len(missing)} missing chunks.'
else:
    rows = []
    segment_lookup = {}
    for idx in tqdm(needed_chunk_indices, desc='PyVi segment needed chunks'):
        text = normalize_text_for_encoder(chunks[idx]['text'], config.max_text_chars)
        seg = pyvi_segment(text)
        segment_lookup[idx] = seg
        rows.append({'chunk_idx': idx, 'chunk_id': chunks[idx]['chunk_id'], 'doc_id': chunks[idx]['doc_id'], 'segmented_text': seg})
    write_jsonl(SEGMENTED_CHUNKS_FILE, rows)

if SEGMENTED_QUERIES_FILE.exists():
    query_segment_lookup = read_json(SEGMENTED_QUERIES_FILE)
else:
    query_segment_lookup = {}
    for payload in [train_payload, dev_payload]:
        for qid, row in tqdm(list(payload.items()), desc='PyVi segment queries'):
            query_segment_lookup[str(qid)] = pyvi_segment(str(row.get('question', '')))
    write_json(SEGMENTED_QUERIES_FILE, query_segment_lookup)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
frozen_model = SentenceTransformer(config.model_name, device=device, trust_remote_code=True)
frozen_model.max_seq_length = config.max_seq_length
param_count = int(sum(p.numel() for p in frozen_model.parameters()))
manifest = {
    'models': [{'model_id': config.model_name, 'role': 'clean_sbert_cl_base_and_finetuned_encoder', 'parameter_count': param_count, 'parameter_count_source': 'sum(p.numel() for p in SentenceTransformer parameters)'}],
    'total_known_parameters': param_count,
    'max_total_parameters': 4_000_000_000,
    'allowed_models': sorted(ALLOWED_MODELS),
    'no_hosted_inference_or_api': True,
    'local_training_and_inference': True,
    'word_segmentation': 'PyVi for query and chunk text',
}
if manifest['total_known_parameters'] >= manifest['max_total_parameters']:
    raise ValueError('Parameter audit failed: total >= 4B')
write_json(OUTPUT_DIR / 'reports' / 'model_manifest.json', manifest)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

chunk_idx_to_position = {idx: pos for pos, idx in enumerate(needed_chunk_indices)}
FROZEN_EMB_FILE = OUTPUT_DIR / 'cache' / 'frozen_needed_chunk_embeddings_fp16.npy'
FROZEN_EMB_META = OUTPUT_DIR / 'cache' / 'frozen_needed_chunk_embeddings_meta.json'
if FROZEN_EMB_FILE.exists() and FROZEN_EMB_META.exists():
    frozen_chunk_embeddings = np.load(FROZEN_EMB_FILE, mmap_mode='r')
    meta = read_json(FROZEN_EMB_META)
    assert meta['num_chunks'] == len(needed_chunk_indices), 'Frozen embedding cache has wrong num_chunks.'
else:
    texts = [segment_lookup[idx] for idx in needed_chunk_indices]
    probe = frozen_model.encode(texts[:1], batch_size=1, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    arr = np.lib.format.open_memmap(FROZEN_EMB_FILE, mode='w+', dtype=np.float16, shape=(len(texts), int(probe.shape[1])))
    for start in tqdm(range(0, len(texts), config.frozen_encode_batch_size), desc='Frozen encode needed chunks'):
        batch = texts[start:start + config.frozen_encode_batch_size]
        vecs = frozen_model.encode(batch, batch_size=config.frozen_encode_batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        arr[start:start + len(batch)] = vecs.astype(np.float16)
    arr.flush()
    write_json(FROZEN_EMB_META, {'model_name': config.model_name, 'num_chunks': len(texts), 'dim': int(probe.shape[1]), 'dtype': 'float16'})
    frozen_chunk_embeddings = np.load(FROZEN_EMB_FILE, mmap_mode='r')

train_qids = list(map(str, train_payload.keys()))
dev_qids = list(map(str, dev_payload.keys()))
all_qids = train_qids + dev_qids
all_qtexts = [query_segment_lookup[qid] for qid in all_qids]
all_qemb = frozen_model.encode(all_qtexts, batch_size=config.eval_batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
query_embedding_lookup = {qid: all_qemb[i].astype(np.float32) for i, qid in enumerate(all_qids)}

def chunk_vec(chunk_idx: int) -> np.ndarray:
    return np.asarray(frozen_chunk_embeddings[chunk_idx_to_position[chunk_idx]], dtype=np.float32)

def cosine_for_query(qid: str, chunk_idx: int) -> float:
    return float(np.dot(query_embedding_lookup[qid], chunk_vec(chunk_idx)))

examples = []
for qid, item in tqdm(train_struct.items(), desc='Select positives and negatives'):
    gold_set = set(item['gold'])
    neg_candidates = [idx for idx in item['negative_candidates'] if chunks[idx]['doc_id'] not in gold_set]
    if not neg_candidates:
        continue
    semantic_pool = sorted(neg_candidates, key=lambda idx: cosine_for_query(qid, idx), reverse=True)[:config.semantic_negative_pool_size]
    lexical_pool = neg_candidates[config.lexical_negative_band_start:config.lexical_negative_band_end] or neg_candidates[:config.lexical_negative_band_end]
    for doc_id, pos_candidates in item['positives_by_doc'].items():
        ranked_pos = sorted(pos_candidates, key=lambda idx: cosine_for_query(qid, idx), reverse=True)
        for pos_idx in ranked_pos[:config.positives_per_gold]:
            examples.append({'query_id': qid, 'segmented_question': query_segment_lookup[qid], 'gold_doc_id': doc_id, 'positive_chunk_idx': pos_idx, 'positive_chunk_id': chunks[pos_idx]['chunk_id'], 'positive_text': segment_lookup[pos_idx], 'lexical_negative_pool': lexical_pool, 'semantic_negative_pool': semantic_pool})

assert examples, 'No training examples were built.'
write_json(OUTPUT_DIR / 'training' / 'mining_report.json', {'num_examples': len(examples), 'positives_per_gold': config.positives_per_gold, 'negative_source': 'step5 fused candidates after removing all gold document IDs'})
write_jsonl(OUTPUT_DIR / 'training' / 'train_examples.jsonl', examples)
print({'train_examples': len(examples), 'frozen_embeddings': frozen_chunk_embeddings.shape})


## Dataset, Dev Eval, And Training Loop


In [ ]:
class LegalContrastiveDataset(Dataset):
    def __init__(self, examples: list[dict[str, Any]], seed: int):
        self.examples = examples
        self.rng = random.Random(seed)
    def __len__(self) -> int:
        return len(self.examples)
    def _pick(self, pool: list[int], offset: int) -> int:
        return pool[(self.rng.randrange(len(pool)) + offset) % len(pool)]
    def __getitem__(self, idx: int) -> dict[str, Any]:
        ex = self.examples[idx]
        lex_idx = self._pick(ex['lexical_negative_pool'], idx)
        sem_idx = self._pick(ex['semantic_negative_pool'], idx * 17)
        return {'query_id': ex['query_id'], 'query': ex['segmented_question'], 'positive': ex['positive_text'], 'lex_negative': segment_lookup[lex_idx], 'sem_negative': segment_lookup[sem_idx], 'positive_doc_id': ex['gold_doc_id'], 'lex_negative_doc_id': chunks[lex_idx]['doc_id'], 'sem_negative_doc_id': chunks[sem_idx]['doc_id']}

def collate_batch(rows: list[dict[str, Any]]) -> dict[str, Any]:
    return {key: [row[key] for row in rows] for key in rows[0]}

def model_embeddings(model: SentenceTransformer, texts: list[str]) -> torch.Tensor:
    features = model.tokenize(texts)
    features = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in features.items()}
    out = model(features)['sentence_embedding']
    return F.normalize(out, p=2, dim=1)

def contrastive_loss(model: SentenceTransformer, batch: dict[str, Any], temperature: float) -> torch.Tensor:
    for i, pos_doc in enumerate(batch['positive_doc_id']):
        if batch['lex_negative_doc_id'][i] == pos_doc or batch['sem_negative_doc_id'][i] == pos_doc:
            raise RuntimeError('Found explicit negative with same doc as positive.')
    q_emb = model_embeddings(model, batch['query'])
    p_emb = model_embeddings(model, batch['positive'])
    lex_emb = model_embeddings(model, batch['lex_negative'])
    sem_emb = model_embeddings(model, batch['sem_negative'])
    logits = torch.cat([q_emb @ p_emb.T, torch.sum(q_emb * lex_emb, dim=1, keepdim=True), torch.sum(q_emb * sem_emb, dim=1, keepdim=True)], dim=1) / temperature
    labels = torch.arange(q_emb.size(0), device=q_emb.device)
    return F.cross_entropy(logits, labels)

def build_dev_eval_candidates():
    eval_candidates = {}
    needed = set()
    for qid, row in tqdm(list(dev_payload.items()), desc='Build dev eval candidates'):
        qid = str(qid)
        question = str(row.get('question', ''))
        rows = []
        for doc_id in dev_rankings[qid][:config.eval_candidate_docs]:
            cand = top_lexical_chunks_for_doc(question, str(doc_id), config.eval_evidence_chunks_per_doc)
            if cand:
                rows.append({'doc_id': str(doc_id), 'chunk_idx': cand[0]})
                needed.add(cand[0])
        eval_candidates[qid] = rows
    return eval_candidates, needed

dev_eval_candidates, dev_eval_needed = build_dev_eval_candidates()
extra = sorted(dev_eval_needed - set(segment_lookup))
if extra:
    with SEGMENTED_CHUNKS_FILE.open('a', encoding='utf-8') as f:
        for idx in tqdm(extra, desc='PyVi segment extra dev chunks'):
            seg = pyvi_segment(normalize_text_for_encoder(chunks[idx]['text'], config.max_text_chars))
            segment_lookup[idx] = seg
            f.write(json.dumps({'chunk_idx': idx, 'chunk_id': chunks[idx]['chunk_id'], 'doc_id': chunks[idx]['doc_id'], 'segmented_text': seg}, ensure_ascii=False) + '\n')

def rerank_dev_with_model(model: SentenceTransformer, split_name: str):
    model.eval()
    rankings = {}
    with torch.no_grad():
        for qid, rows in tqdm(dev_eval_candidates.items(), desc=f'{split_name}: rerank dev candidates'):
            c_texts = [segment_lookup[row['chunk_idx']] for row in rows]
            if not c_texts:
                rankings[qid] = []
                continue
            q_emb = model_embeddings(model, [query_segment_lookup[qid]])
            scores = []
            for start in range(0, len(c_texts), config.eval_batch_size):
                c_emb = model_embeddings(model, c_texts[start:start + config.eval_batch_size])
                scores.extend((c_emb @ q_emb.T).squeeze(1).detach().cpu().numpy().tolist())
            doc_scores = [(rows[i]['doc_id'], float(scores[i]), i) for i in range(len(rows))]
            doc_scores.sort(key=lambda x: (-x[1], x[2]))
            rankings[qid] = [doc_id for doc_id, _score, _i in doc_scores]
    return rankings, evaluate_rankings(rankings, dev_payload)

del frozen_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = SentenceTransformer(config.model_name, device=device, trust_remote_code=True)
model.max_seq_length = config.max_seq_length
train_loader = DataLoader(LegalContrastiveDataset(examples, config.seed), batch_size=config.train_batch_size, shuffle=True, drop_last=True, collate_fn=collate_batch, num_workers=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
total_steps = len(train_loader) * config.epochs
warmup_steps = max(1, int(total_steps * config.warmup_ratio))
def lr_lambda(step: int) -> float:
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    return float(max(0, total_steps - step)) / float(max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

MODEL_DIR = OUTPUT_DIR / 'models' / 'bkai_sbert_cl_clean'
BEST_MODEL_DIR = OUTPUT_DIR / 'models' / 'bkai_sbert_cl_clean_best'
for path in [MODEL_DIR, BEST_MODEL_DIR]:
    if path.exists():
        shutil.rmtree(path)

history = []
best_key = (-1.0, -1.0, -1.0)
global_step = 0
started = time.time()
base_rankings, base_metrics = rerank_dev_with_model(model, 'base_bkai_pyvi')
write_json(OUTPUT_DIR / 'metrics' / 'dev_metrics_base_bkai_pyvi.json', base_metrics)
write_jsonl(OUTPUT_DIR / 'rankings' / 'dev_rankings_base_bkai_pyvi.jsonl', ({'query_id': qid, 'doc_ids': docs, 'gold': gold_docs(dev_payload[qid])} for qid, docs in base_rankings.items()))
print('base dev:', json.dumps(base_metrics['macro'], ensure_ascii=False, indent=2))

for epoch in range(1, config.epochs + 1):
    model.train()
    running = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{config.epochs}')
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            loss = contrastive_loss(model, batch, config.temperature)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        global_step += 1
        running.append(float(loss.detach().cpu()))
        running = running[-100:]
        pbar.set_postfix({'loss': sum(running) / len(running), 'lr': scheduler.get_last_lr()[0]})
    dev_rankings_epoch, dev_metrics_epoch = rerank_dev_with_model(model, f'epoch_{epoch}')
    macro = dev_metrics_epoch['macro']
    key = (macro.get(config.best_metric, 0.0), macro.get('precision@5', 0.0), macro.get('mrr', 0.0))
    row = {'epoch': epoch, 'global_step': global_step, 'train_loss_last100': sum(running) / max(1, len(running)), 'dev_macro': macro, 'seconds_elapsed': round(time.time() - started, 3)}
    history.append(row)
    write_json(OUTPUT_DIR / 'metrics' / 'training_history.json', history)
    write_jsonl(OUTPUT_DIR / 'rankings' / f'dev_rankings_epoch_{epoch}.jsonl', ({'query_id': qid, 'doc_ids': docs, 'gold': gold_docs(dev_payload[qid])} for qid, docs in dev_rankings_epoch.items()))
    print('epoch metrics:', json.dumps(row, ensure_ascii=False, indent=2))
    if key > best_key:
        best_key = key
        model.save(str(BEST_MODEL_DIR))
        write_json(OUTPUT_DIR / 'metrics' / 'best_epoch.json', {'epoch': epoch, 'selection_key': list(best_key), 'dev_macro': macro})
        print('Saved best model:', BEST_MODEL_DIR)

model.save(str(MODEL_DIR))
print('Saved final model:', MODEL_DIR)
print('Saved best model:', BEST_MODEL_DIR)


## Final Reports And Artifact Zip


In [ ]:
def model_file_checksums(model_dir: Path) -> dict[str, str]:
    out = {}
    for path in sorted(model_dir.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.bin', '.safetensors', '.json', '.txt', '.model', '.codes'}:
            out[str(path.relative_to(model_dir)).replace('\\', '/')] = sha256_file(path)
    return out

best_epoch = read_json(OUTPUT_DIR / 'metrics' / 'best_epoch.json')
run_report = {
    'step': 'step7b_1_clean_sbert_cl_train',
    'status': 'completed',
    'purpose': 'clean PyVi SBERT-CL checkpoint for Step 7b.2 global dense retrieval',
    'inputs': {name: str(path) for name, path in required.items()},
    'config': asdict(config),
    'best_epoch': best_epoch,
    'model_dir': 'models/bkai_sbert_cl_clean_best',
    'model_checksums': model_file_checksums(BEST_MODEL_DIR),
    'outputs_for_next_checkpoint': ['checkpoint1/models/bkai_sbert_cl_clean_best', 'checkpoint1/configs/step7b1_config.json', 'checkpoint1/reports/model_manifest.json', 'checkpoint1/reports/run_report.json', 'checkpoint1/metrics/best_epoch.json'],
    'no_submission_created': True,
    'reason_no_submission': 'Checkpoint 1 only trains the encoder. Step 7b.2 must run global dense retrieval/RRF before public submission.',
}
write_json(OUTPUT_DIR / 'reports' / 'run_report.json', run_report)

zip_path = Path('/kaggle/working/step7b_checkpoint1.zip')
if zip_path.exists():
    zip_path.unlink()
include_dirs = [OUTPUT_DIR / 'configs', OUTPUT_DIR / 'metrics', OUTPUT_DIR / 'reports', BEST_MODEL_DIR]
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for base in include_dirs:
        for path in sorted(base.rglob('*')):
            if path.is_file():
                arc = Path('checkpoint1') / path.relative_to(OUTPUT_DIR)
                zf.write(path, arc.as_posix())

print('Artifact zip:', zip_path)
print('Best epoch:', json.dumps(best_epoch, ensure_ascii=False, indent=2))
print('Download this zip only if Step 7b.2 will run in a new Kaggle session/dataset.')
